# Example 2 — Full Pipeline with Google Earth Engine

`run_crop_stage_from_gee` is the single-polygon entry point for GEE users:
provide a polygon (any supported format) and a lookback window, and it returns
the crop stage for the last available observation.

Both `fetch_ndvi` and `run_crop_stage_from_gee` work on **one polygon at a time**.
For multiple fields see Section 5 — it shows a ThreadPoolExecutor + tqdm pattern
you can drop straight into production.

This notebook also walks through the same pipeline step by step (Sections 2–4)
to show what happens under the hood.

### Prerequisites

1. A GEE account — sign up at https://earthengine.google.com
2. Install both requirements files: `pip install -r requirements.txt -r requirements-gee.txt`
3. Authenticate once: `earthengine authenticate` in your terminal

Call `ee.Initialize()` **before** importing `gee_fetch`.

In [ ]:
import sys
sys.path.insert(0, "../src")

import ee
ee.Initialize()   # <-- must come before gee_fetch import

import geopandas as gpd
import shapely.geometry
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

from gee_fetch import fetch_ndvi, run_crop_stage_from_gee
from crop_stage import smooth_daily_interpolate_ndvi, estimate_stage_adaptive

## 1. Define a polygon

`fetch_ndvi` and `run_crop_stage_from_gee` accept the field polygon in any of
the formats shown below. Files and GeoDataFrames with a UTM CRS are also
supported — the polygon is reprojected to WGS84 internally.

In [ ]:
# All of the following represent the same field — pick whichever fits your workflow.

# A — list of [lon, lat] coordinate pairs
#     Order is [longitude, latitude] — NOT [lat, lon].
#     WGS84 required. Ring may be open or closed. UTM not supported.
polygon_as_list = [
    [-77.897937, 35.571295],
    [-77.898152, 35.570355],
    [-77.897400, 35.569852],
    [-77.895398, 35.569557],
    [-77.895062, 35.570803],
    [-77.897937, 35.571295],  # close the ring (optional)
]

# B — GeoJSON dict (Feature, FeatureCollection, or bare Geometry)
#     WGS84 by spec (RFC 7946). UTM not supported for this format.
polygon_as_geojson = {
    "type": "Feature",
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [-77.897937, 35.571295],
            [-77.898152, 35.570355],
            [-77.897400, 35.569852],
            [-77.895398, 35.569557],
            [-77.895062, 35.570803],
            [-77.897937, 35.571295],
        ]]
    },
    "properties": {}
}

# C — shapely Geometry
#     WGS84 inferred if bounds look like lon/lat (-180..180, -90..90).
#     UTM not supported — shapely objects carry no CRS.
polygon_as_shapely = shapely.geometry.Polygon([
    (-77.897937, 35.571295), (-77.898152, 35.570355),
    (-77.897400, 35.569852), (-77.895398, 35.569557),
    (-77.895062, 35.570803),
])

# D — GeoDataFrame
#     CRS is read from the .crs attribute. UTM is supported — set crs accordingly.
#     Example UTM: gpd.GeoDataFrame(geometry=[geom], crs="EPSG:32617")
polygon_as_gdf = gpd.GeoDataFrame(geometry=[polygon_as_shapely], crs="EPSG:4326")

# E — file path: GeoJSON, Shapefile (.shp), zipped Shapefile (.zip),
#     GeoPackage (.gpkg), KML (.kml), KMZ (.kmz)
#     CRS is read from the file. UTM shapefiles and GeoPackages are supported.
polygon_as_file    = "../sample_data/sample_field.geojson"
polygon_as_shp     = "../sample_data/sample_field.shp"
polygon_as_kml     = "../sample_data/sample_field.kml"
polygon_as_kmz     = "../sample_data/sample_field.kmz"
polygon_as_gpkg_utm = "../sample_data/sample_field_utm.gpkg"   # UTM example
polygon_as_zip_utm  = "../sample_data/sample_field_utm.zip"    # UTM zipped SHP

# F — ee.Geometry (e.g. from a BigQuery/GEE pipeline)
#     Coordinates are retrieved via .getInfo() — a synchronous GEE API call.
#     When performance matters, pass the original polygon format above instead.
# polygon_as_ee = ee.Geometry.Polygon(polygon_as_list)

In [ ]:
# Choose the format you want to use — the rest of the notebook uses polygon_as_list.
polygon = polygon_as_list

START_DATE = "2023-03-01"
END_DATE   = "2023-11-30"

## 2. Fetch NDVI from GEE

In [ ]:
ndvi_df = fetch_ndvi(
    polygon,
    start_date=START_DATE,
    end_date=END_DATE,
    poly_name="example_field",
    buffer_m=-10,     # inset 10 m to avoid boundary pixels
)

print(f"Retrieved {len(ndvi_df)} observations")
ndvi_df.head()

## 3. Smooth and estimate stage

In [ ]:
if ndvi_df.empty:
    raise ValueError("No valid NDVI observations returned — polygon may be too cloudy, too small, or outside GEE coverage.")

df_smooth = smooth_daily_interpolate_ndvi(ndvi_df)
result = estimate_stage_adaptive(
    df_smooth["NDVI_smooth"].to_numpy(),
    dates=df_smooth["date"],
)

for k, v in result.items():
    print(f"  {k:20s}: {v}")

## 4. Visualise

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for sensor, grp in ndvi_df.groupby("sensor"):
    ax.scatter(grp["date"], grp["NDVI"], alpha=0.5, s=35, label=sensor)

ax.plot(df_smooth["date"], df_smooth["NDVI_smooth"], color="steelblue", lw=2, label="Smoothed")
ax.axhline(result["Lower_threshold"], color="orange", ls="--",
           label=f"Lower ({result['Lower_threshold']:.2f})")
ax.axhline(result["Upper_threshold"], color="green",  ls="--",
           label=f"Upper ({result['Upper_threshold']:.2f})")

if result.get("Peak_date"):
    ax.axvline(result["Peak_date"], color="purple", ls=":",
               label=f"Peak ({result['Peak_date'].strftime('%d-%b-%Y')})")

ax.set_ylim(0, 1)
ax.set_ylabel("NDVI")
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(mdates.AutoDateLocator()))
ax.legend(loc="upper left", fontsize=8)
ax.set_title(f"Stage: {result['Stage']} — {result['Stage_description']}")
plt.tight_layout()
plt.show()

## 5. Multiple polygons

Both `fetch_ndvi` and `run_crop_stage_from_gee` work on one polygon at a time.
For multiple fields, loop or use a `ThreadPoolExecutor`.

The example below runs all five polygon formats from Section 1 in parallel.
Since they all represent the same field, all results should be identical —
a quick sanity check that every format works end-to-end.

In [ ]:
polygon_formats = {
    "list":         polygon_as_list,
    "geojson_dict": polygon_as_geojson,
    "shapely":      polygon_as_shapely,
    "gdf":          polygon_as_gdf,
    "geojson_file": polygon_as_file,
    "shp":          polygon_as_shp,
    "kml":          polygon_as_kml,
    "kmz":          polygon_as_kmz,
    "gpkg_utm":     polygon_as_gpkg_utm,
    "zip_utm":      polygon_as_zip_utm,
}

def _fetch_one(fmt_name, polygon):
    result = run_crop_stage_from_gee(polygon, lookback_days=270)
    return {
        "format":            fmt_name,
        "Stage":             result["Stage"],
        "Stage_description": result["Stage_description"],
        "Peak_date":         result.get("Peak_date"),
        "Days_since_peak":   result.get("Days_since_peak"),
    }

with ThreadPoolExecutor(max_workers=len(polygon_formats)) as executor:
    futures = {
        executor.submit(_fetch_one, name, poly): name
        for name, poly in polygon_formats.items()
    }
    results = []
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing"):
        results.append(future.result())

pd.DataFrame(results).sort_values("format").reset_index(drop=True)